In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("talabat_enhanced_orders.csv")
df_fe = df.copy()

df_fe["Order_Time"] = pd.to_datetime(df_fe["Order_Time"], errors="coerce")
df_fe["order_hour"] = df_fe["Order_Time"].dt.hour
df_fe["order_dayofweek"] = df_fe["Order_Time"].dt.dayofweek
df_fe["is_weekend"] = df_fe["order_dayofweek"].isin([5,6]).astype(int)

df_fe["is_peak_hour"] = df_fe["order_hour"].isin(list(range(12,16)) + list(range(19,24))).astype(int)

df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]
df_fe["price_tier"] = pd.cut(
    df_fe["Total_Price"],
    bins=[0, 100, 250, 500, np.inf],
    labels=["low","medium","high","very_high"]
)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df_fe["haversine_rest_to_cust_km"] = haversine_km(
    df_fe["Restaurant_Lat"], df_fe["Restaurant_Lon"],
    df_fe["Customer_Lat"], df_fe["Customer_Lon"]
)

In [3]:
# Task 1: create one new engineered feature
df_fe["price_per_km"] = df_fe["Total_Price"] / (df_fe["Delivery_Distance_km"] + 0.001)

print("Task 1 complete: 'price_per_km' added.")
df_fe[["Total_Price", "Delivery_Distance_km", "price_per_km"]].head()

Task 1 complete: 'price_per_km' added.


,Total_Price,Delivery_Distance_km,price_per_km
0,273.72,1.666106,164.188692
1,365.82,2.738698,133.525689
2,401.94,2.929079,137.177202
3,221.18,0.677498,325.984776
4,355.55,1.994769,178.151903


This feature captures the delivery cost relative to distance, which may help the model distinguish between short expensive deliveries and long inexpensive ones.

In [5]:
# Task 2: Tighter peak hour rule
df_fe["is_peak_hour_tight"] = df_fe["order_hour"].isin([13, 14, 20, 21]).astype(int)

print("Original peak hour counts:")
print(df_fe["is_peak_hour"].value_counts())
print("\nTight peak hour counts:")
print(df_fe["is_peak_hour_tight"].value_counts())

Original peak hour counts:
is_peak_hour
0    62701
1    37299
Name: count, dtype: int64

Tight peak hour counts:
is_peak_hour_tight
0    83527
1    16473
Name: count, dtype: int64


In [6]:
# Task 3: Change top_k to 4 to actually see a difference
top_k = 4
top_items = df_fe["Item_Name"].value_counts().head(top_k).index
df_fe["Item_Name_reduced"] = np.where(df_fe["Item_Name"].isin(top_items), df_fe["Item_Name"], "Other")

print("Value counts for top_k=4:")
print(df_fe["Item_Name_reduced"].value_counts())

drop_cols = [
    "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
    "Order_Time", "Delivery_Time", "Delivery_Duration_Minutes",
    "Item_Name", "is_peak_hour"
]
drop_cols = [c for c in drop_cols if c in df_fe.columns]

X = df_fe.drop(columns=drop_cols + ["Order_Status"])
y = df_fe["Order_Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

Value counts for top_k=4:
Item_Name_reduced
Other            55151
Shawarma         11320
Pizza            11229
Fried Chicken    11171
Burger           11129
Name: count, dtype: int64


In [8]:
# Task 4: Feature Selection using SelectFromModel
from sklearn.feature_selection import SelectFromModel

rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample")
model_baseline = Pipeline(steps=[("preprocess", preprocess), ("rf", rf_baseline)])
model_baseline.fit(X_train, y_train)
y_pred_base = model_baseline.predict(X_test)
print("Accuracy (Baseline):", round(accuracy_score(y_test, y_pred_base), 4))

selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", selector),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample"))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Accuracy (with Feature Selection):", round(accuracy_score(y_test, y_pred_fs), 4))

Accuracy (Baseline): 0.8519
Accuracy (with Feature Selection): 0.8519
